In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.ensemble import RandomForestClassifier

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\jayes\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [ ]:
# MLP Classifier using PyTorch for classification tasks. It supports multiple hidden layers and uses ReLU activation by default.
class MLPClassifierTorch:
    def __init__(self, input_size, num_classes, hidden_layers=(128, 64), activation=torch.relu, lr=1e-3, device=None):
        self.input_size = input_size
        self.num_classes = num_classes
        self.hidden_layers = hidden_layers
        self.activation = activation
        self.lr = lr
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")

        layers = []
        in_features = input_size
        for h in hidden_layers:
            layers.append(torch.nn.Linear(in_features, h))
            layers.append(torch.nn.ReLU())
            in_features = h
        layers.append(torch.nn.Linear(in_features, num_classes))
        self.net = torch.nn.Sequential(*layers).to(self.device)
        self.loss_fn = torch.nn.CrossEntropyLoss()
        self.opt = torch.optim.Adam(self.net.parameters(), lr=self.lr)

    def fit(self, X_train, y_train, epochs=20, batch_size=32):
        X = torch.tensor(X_train, dtype=torch.float32).to(self.device)
        y = torch.tensor(y_train, dtype=torch.long).to(self.device)
        dataset = torch.utils.data.TensorDataset(X, y)
        loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
        self.net.train()
        for _ in range(epochs):
            for xb, yb in loader:
                self.opt.zero_grad()
                logits = self.net(xb)
                loss = self.loss_fn(logits, yb)
                loss.backward()
                self.opt.step()

    def predict(self, X_test):
        self.net.eval()
        X = torch.tensor(X_test, dtype=torch.float32).to(self.device)
        with torch.no_grad():
            logits = self.net(X)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
        return preds

    def score(self, X_test, y_test):
        preds = self.predict(X_test)
        return (preds == y_test).mean()


In [ ]:
# load the dataset
df = pd.read_csv("../datasets/train_waveforms.csv")
df.head()

NameError: name 'pd' is not defined

In [ ]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, X, y=None, transform=None, device=None, x_dtype=torch.float32, y_dtype=torch.long):
        # accept pandas, numpy, torch, lists
        if isinstance(X, pd.DataFrame) or isinstance(X, pd.Series):
            X = X.values
        if isinstance(y, (pd.DataFrame, pd.Series)):
            y = y.values.ravel()

        if not isinstance(X, torch.Tensor):
            X = torch.tensor(np.asarray(X), dtype=x_dtype)
        if y is not None and not isinstance(y, torch.Tensor):
            y = torch.tensor(np.asarray(y), dtype=y_dtype)

        self.device = device or (torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))
        # keep tensors on CPU by default to avoid forcing GPU usage; move if explicitly requested
        self.X = X.to(self.device) if device is not None else X
        self.y = y.to(self.device) if (y is not None and device is not None) else y
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx] if self.y is not None else None
        if self.transform:
            x = self.transform(x)
        return (x, y) if y is not None else x

    def to_dataloader(self, batch_size=32, shuffle=True, num_workers=0, pin_memory=False, drop_last=False):
        return DataLoader(self, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers,
                          pin_memory=pin_memory, drop_last=drop_last)

    def class_counts(self):
        if self.y is None:
            return None
        vals, counts = torch.unique(self.y, return_counts=True)
        return dict(zip(vals.cpu().numpy().tolist(), counts.cpu().numpy().tolist()))

    @property
    def num_features(self):
        return int(self.X.shape[1]) if self.X.ndim > 1 else 1

    def to_numpy(self):
        X_np = self.X.cpu().numpy() if isinstance(self.X, torch.Tensor) else np.asarray(self.X)
        y_np = None
        if self.y is not None:
            y_np = self.y.cpu().numpy() if isinstance(self.y, torch.Tensor) else np.asarray(self.y)
        return X_np, y_np

    def __repr__(self):
        return f"CustomDataset(len={len(self)}, features={self.num_features}, labeled={self.y is not None})"